In [1]:
import pandas as pd
import numpy as np
from collections import defaultdict
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

In [2]:
# Load Data
album_df  = pd.read_csv('Assets/CSV/album_data.csv')
artist_df = pd.read_csv('Assets/CSV/artist_data.csv')
genre_df  = pd.read_csv('Assets/CSV/genre_data.csv')
track_df  = pd.read_csv('Assets/CSV/track_data.csv')
train_df  = pd.read_csv('Assets/CSV/train_data.csv')
test_df   = pd.read_csv('Assets/CSV/test_data.csv')

print(f"Albums  : {len(album_df):>10,} rows")
print(f"Artists : {len(artist_df):>10,} rows")
print(f"Genres  : {len(genre_df):>10,} rows")
print(f"Tracks  : {len(track_df):>10,} rows")
print(f"Train   : {len(train_df):>10,} rows")
print(f"Test    : {len(test_df):>10,} rows  ({test_df['UserID'].nunique():,} unique users)")

Albums  :     52,829 rows
Artists :     18,674 rows
Genres  :        567 rows
Tracks  :    224,041 rows
Train   : 12,403,575 rows
Test    :    120,000 rows  (20,000 unique users)


# Part 1: Heuristic-Based Music Recommendation

## a) Feature Engineering & Statistical Aggregation

In [3]:
print("=== TRAIN DATA (ratings) ===")
display(train_df.head(8))
print(f"\nRating range: {train_df['Rating'].min()} - {train_df['Rating'].max()}")
print(f"Rating mean : {train_df['Rating'].mean():.2f}")
print(f"Rating std  : {train_df['Rating'].std():.2f}")

print("\n=== TEST DATA (candidates to label) ===")
display(test_df.head(12))
print(f"\nTracks per user: {test_df.groupby('UserID').size().unique()}")  # Should always be [6]

print("\n=== TRACK HIERARCHY ===")
display(track_df.head(6))

# Find tracks missing genre
genre_cols = [c for c in track_df.columns if c.startswith('Genre')]
genre_fill = track_df[genre_cols].notna().sum(axis=1)
print(f"\nGenres per track — min: {genre_fill.min()}, max: {genre_fill.max()}, mean: {genre_fill.mean():.2f}")

# Find tracks missing album or artist
print(f"Tracks missing AlbumID : {track_df['AlbumID'].isna().sum():,}")
print(f"Tracks missing ArtistID: {track_df['ArtistID'].isna().sum():,}")

=== TRAIN DATA (ratings) ===


,UserID,ItemID,Rating
0,199808,248969,90
1,199808,2663,90
2,199808,28341,90
3,199808,42563,90
4,199808,59092,90
5,199808,64052,90
6,199808,69022,90
7,199808,77710,90



Rating range: 0 - 100
Rating mean : 49.77
Rating std  : 38.03

=== TEST DATA (candidates to label) ===


,UserID,TrackID
0,199810,208019
1,199810,74139
2,199810,9903
3,199810,242681
4,199810,18515
5,199810,105760
6,199812,276940
7,199812,142408
8,199812,130023
9,199812,29189



Tracks per user: [6]

=== TRACK HIERARCHY ===


,TrackID,AlbumID,ArtistID,Genre1,Genre2,Genre3,Genre4,Genre5,Genre6,Genre7,Genre8,Genre9,Genre10,Genre11,Genre12,Genre13,Genre14,Genre15,Genre16,Genre17,Genre18,Genre19,Genre20,Genre21
0,1,106710.00,281667.00,214765.00,162234.00,155788.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,280977.00,233685.00,131552.00,173467.00,48505.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,38422.00,219136.00,61215.00,201738.00,88853.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,119529.00,166863.00,17453.00,35389.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,16742.00,294690.00,61215.00,34486.00,274088.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,7,101746.00,44649.00,198263.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Genres per track — min: 0, max: 21, mean: 2.44
Tracks missing AlbumID : 18,509
Tracks missing ArtistID: 26,896


#### Build Track to Heirarchy lookup

In [4]:
# Maps each TrackID (str) -> {'album': str|None, 'artist': str|None, 'genres': [str, ...]}
genre_cols = [c for c in track_df.columns if c.startswith('Genre')]

track_to_hierarchy = {}

for _, row in tqdm(track_df.iterrows(), total=len(track_df), desc="Building hierarchy"):
    track_id  = str(int(row['TrackID']))
    album_id  = str(int(row['AlbumID']))  if pd.notna(row['AlbumID'])  else None
    artist_id = str(int(row['ArtistID'])) if pd.notna(row['ArtistID']) else None
    genres    = [str(int(row[c])) for c in genre_cols if pd.notna(row[c])]

    track_to_hierarchy[track_id] = {
        'album' : album_id,
        'artist': artist_id,
        'genres': genres
    }

print(f"Hierarchy built for {len(track_to_hierarchy):,} tracks")

# Spot-check
sample_id = list(track_to_hierarchy.keys())[0]
print(f"\nSample track {sample_id}: {track_to_hierarchy[sample_id]}")

Building hierarchy:   0%|          | 0/224041 [00:00<?, ?it/s]

Hierarchy built for 224,041 tracks

Sample track 1: {'album': '106710', 'artist': '281667', 'genres': ['214765', '162234', '155788']}


#### Build User to Rating Lookup

In [5]:
# Maps each UserID (str) -> {ItemID (str): Rating (float)}
# ItemID is flat — same dict holds album, artist, genre, and track ratings
user_to_ratings = defaultdict(dict)

for _, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Building rating lookup"):
    user_to_ratings[str(int(row['UserID']))][str(int(row['ItemID']))] = float(row['Rating'])

print(f"Rating lookup built for {len(user_to_ratings):,} users")

# Spot-check
sample_user = list(user_to_ratings.keys())[0]
sample_items = list(user_to_ratings[sample_user].items())[:5]
print(f"\nUser {sample_user} — sample ratings: {sample_items}")

Building rating lookup:   0%|          | 0/12403575 [00:00<?, ?it/s]

Rating lookup built for 49,204 users

User 199808 — sample ratings: [('248969', 90.0), ('2663', 90.0), ('28341', 90.0), ('42563', 90.0), ('59092', 90.0)]


#### If user never rated track

In [6]:
# When a user has never rated a specific album/artist/genre,
# we fall back to the global average rating for that item.
# If the item has never been rated by anyone, we fall back to the overall mean.

overall_mean = train_df['Rating'].mean()
print(f"Overall mean rating: {overall_mean:.2f}")

item_avg_ratings = (
    train_df.groupby('ItemID')['Rating']
    .mean()
    .rename(lambda x: str(int(x)))
    .to_dict()
)

print(f"Global averages computed for {len(item_avg_ratings):,} items")

Overall mean rating: 49.77
Global averages computed for 295,799 items


#### Feature Engineering

In [7]:
# For each (user, track) pair, we produce 7 features
def build_feature_vector(user_id, track_id, track_to_hierarchy, user_to_ratings, item_avg_ratings, overall_mean, default=None):
    """
    Returns a dict with:
        album_score    user's preference score for the track's album
        artist_score   user's preference score for the track's artist
        genre_count    number of genres associated with this track
        genre_max      highest genre preference score
        genre_min      lowest genre preference score
        genre_mean     average genre preference score
        genre_var      variance across genre preference scores

    Lookup priority for each score:
        1. User's own rating for that item (most personal signal)
        2. Global average rating for that item (community signal)
        3. overall_mean / default (cold-start fallback)
    """
    if default is None:
        default = overall_mean

    uid = str(int(user_id))
    tid = str(int(track_id))

    hierarchy = track_to_hierarchy.get(tid, {})
    user_ratings = user_to_ratings.get(uid, {})

    album_id  = hierarchy.get('album')
    artist_id = hierarchy.get('artist')
    genre_ids = hierarchy.get('genres', [])

    # --- Album score ---
    if album_id:
        album_score = user_ratings.get(album_id,
                      item_avg_ratings.get(album_id, default))
    else:
        album_score = default

    # --- Artist score ---
    if artist_id:
        artist_score = user_ratings.get(artist_id,
                       item_avg_ratings.get(artist_id, default))
    else:
        artist_score = default

    # --- Genre scores (one per genre associated with the track) ---
    genre_scores = [
        user_ratings.get(g, item_avg_ratings.get(g, default))
        for g in genre_ids
    ]

    # --- Genre statistics ---
    if genre_scores:
        genre_count = len(genre_scores)
        genre_max   = float(np.max(genre_scores))
        genre_min   = float(np.min(genre_scores))
        genre_mean  = float(np.mean(genre_scores))
        genre_var   = float(np.var(genre_scores))
    else:
        # Track has no genre data at all
        genre_count = 0
        genre_max   = default
        genre_min   = default
        genre_mean  = default
        genre_var   = 0.0

    return {
        'album_score' : album_score,
        'artist_score': artist_score,
        'genre_count' : genre_count,
        'genre_max'   : genre_max,
        'genre_min'   : genre_min,
        'genre_mean'  : genre_mean,
        'genre_var'   : genre_var,
    }

#### Feature Matrix for all test pairs

In [8]:
records = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Building feature matrix"):
    fv = build_feature_vector(
        user_id          = row['UserID'],
        track_id         = row['TrackID'],
        track_to_hierarchy = track_to_hierarchy,
        user_to_ratings  = user_to_ratings,
        item_avg_ratings = item_avg_ratings,
        overall_mean     = overall_mean,
    )
    fv['UserID']  = row['UserID']
    fv['TrackID'] = row['TrackID']
    records.append(fv)

features_df = pd.DataFrame(records, columns=[
    'UserID', 'TrackID',
    'album_score', 'artist_score',
    'genre_count', 'genre_max', 'genre_min', 'genre_mean', 'genre_var'
])

print(f"Feature matrix shape: {features_df.shape}")
display(features_df.head(12))

Building feature matrix:   0%|          | 0/120000 [00:00<?, ?it/s]

Feature matrix shape: (120000, 9)


,UserID,TrackID,album_score,artist_score,genre_count,genre_max,genre_min,genre_mean,genre_var
0,199810,208019,42.76,49.77,0,49.77,49.77,49.77,0.00
1,199810,74139,73.82,43.93,7,80.00,17.14,46.47,347.92
2,199810,9903,49.77,49.77,4,56.01,26.76,39.40,115.81
3,199810,242681,56.08,67.74,3,69.53,50.62,62.04,67.33
4,199810,18515,58.01,70.00,3,55.55,6.36,32.66,408.94
5,199810,105760,56.42,90.00,4,80.00,50.52,66.52,184.98
6,199812,276940,46.71,41.89,1,46.85,46.85,46.85,0.00
7,199812,142408,100.00,100.00,6,80.00,34.37,61.60,272.57
8,199812,130023,100.00,100.00,4,80.00,34.37,57.61,501.84
9,199812,29189,56.92,76.71,6,80.00,39.76,55.81,217.21


Heirarchy:
Track (a specific song)
belongs to

Album (collection of tracks)
made by

Artist (musician)
categorized by

Genres (can be several or one genre)

When we test if a user will like a track, we check all three levels:
Album Score:
- Has user rated album the track belongs to:
-- Yes: Use that rating
-- No: User global average
-- No one rated: User overall mean
we then continue for artist this way

We do the same for genre score, but we also generate statistics:
genre_count: (track has how many genres)
genre_max: (highest score from a specific genre it belongs to)
genre_min: (lowest genre score)
genre_mean: (average)
genre_var: (variance, a high variance indicates user loves some genres but hates others)

## b) Decision Logic & Rule Definition

#### Weighted Hierarchical Average
The heirarchy creates a ladder, where album is the most specific signal, then the artist, then the genre as the broadest category. Weighting by the specific information available will help match a track to a user as closely as possible:
final_score = 0.40 × album_score + 0.30 × artist_score + 0.30 × genre_mean

As album rating is the strongest (we give it the highest weight of 0.4)
We equate the score given to the artist and the genre_mean to be 0.3

#### Maximum Genre Score
A specific track may have several genres, and the user may have difference preferences for different genres. So if a track has a specific genre the user really likes, it is likely the user will like that track. This can also be a good rule towards users with few album and/or artist ratings but strong genre prefernces. 
final_score = 0.70 × genre_max + 0.20 × artist_score + 0.10 × album_score

So we give a large weight to the maximum genre score, and a lower weight towards the artist and album, we give a higher score to the artist as opposed to the album, as a artist is likely to create music related to a specific genre.

These two strategies will likely create different ratings towards tracks for a specific user, as strategy 1 recommends based on mainly album and artist values, while strategy 2 deprioritizes the very qualities strategy 1 favors, and will prioritize tracks with known genre values

In [9]:
def score_strategy_1(row):
    """Weighted Hierarchical Average: album 40%, artist 30%, genre_mean 30%"""
    return (
        0.40 * row['album_score'] +
        0.30 * row['artist_score'] +
        0.30 * row['genre_mean']
    )

def score_strategy_2(row):
    """Maximum Genre Score: genre_max 70%, artist 20%, album 10%"""
    return (
        0.70 * row['genre_max'] +
        0.20 * row['artist_score'] +
        0.10 * row['album_score']
    )

In [10]:
# Generate predictions
def generate_predictions(features_df, score_col):
    """
    For each user, rank their 6 tracks by score.
    Top 3: label 1, bottom 3: label 0.
    """
    predictions = []

    for user_id, group in features_df.groupby('UserID'):
        ranked = group.sort_values(score_col, ascending=False).reset_index(drop=True)

        for i, row in ranked.iterrows():
            predictions.append({
                'TrackID'  : f"{int(user_id)}_{int(row['TrackID'])}",
                'Predictor': 1 if i < 3 else 0
            })

    return pd.DataFrame(predictions)


# Generate both submissions
features_df['score_s1'] = features_df.apply(score_strategy_1, axis=1)
features_df['score_s2'] = features_df.apply(score_strategy_2, axis=1)

submission_s1 = generate_predictions(features_df, 'score_s1')
submission_s2 = generate_predictions(features_df, 'score_s2')

submission_s1.to_csv('submission_strategy1_weighted_avg.csv', index=False)
submission_s2.to_csv('submission_strategy2_max_genre.csv', index=False)

# Validation: each user must have exactly 3 ones and 3 zeros
for name, df in [("S1", submission_s1), ("S2", submission_s2)]:
    df['uid'] = df['TrackID'].str.split('_').str[0]
    check = df.groupby('uid')['Predictor'].sum()
    errors = (check != 3).sum()
    print(f"{name}: {errors} users with incorrect label counts")


S1: 0 users with incorrect label counts
S2: 0 users with incorrect label counts


Scores:
weighted_avg strategy: 0.759
max_genre strategy: 0.704

# Part 2: Addressing the Cold Start Problem

## a) Strategy Design (The Global Fallback)

A cold start user is identified as someone who:
1. has zero ratings in train_data.csv 
2. Has ratings, but one match any album, artist, or genre

In both of these cases, or feature vector returns the overall_mean (50) for every single feature, which will make all 6 tracks identical.

Instead of defaulting to a overall mean, we use global popularity, as in, which albums/aritists/genres do users in general rate highly

Using just the global average still introduces a problem, an album rated just one time at a score of 95, looks more favorable than an album rated 50 times at a score of 85. So we can introduce Bayesian average to solve this issue.

Bayesian average is a statsitcal method that calculates a weighted average that takes into account prior knowledge (history) to evaulaute the data. This prevents items with few and extreme ratings from dominating the ranks, by shrinking their average towards the global mean.

Bayesian Average = (C*M + S*V)/(C*V) 
S: Specific item's average rating (current item)
V: Number of reviews for that item

M: The global average (of all items)
C: Confidence Threshold (Average number of reviews per item)

In [11]:
# Compute global popularity scores

item_stats = (
    train_df.groupby('ItemID')['Rating']
    .agg(['mean', 'count'])
    .reset_index()
)

item_stats.columns = ['ItemID', 'global_mean', 'rating_count']
item_stats['ItemID'] = item_stats['ItemID'].astype(str)

# Bayesian average: items with fewer ratings get pulled toward the overall mean.
C = item_stats['rating_count'].median()   # confidence threshold (median count)
m = overall_mean                          # prior = overall dataset mean
item_stats['bayesian_avg'] = (
    (item_stats['rating_count'] * item_stats['global_mean'] + C * m) /
    (item_stats['rating_count'] + C)
)

# Lookup dict: item_id -> bayesian popularity score
global_item_scores = dict(zip(item_stats['ItemID'], item_stats['bayesian_avg']))
print(f"Global popularity scores computed for {len(global_item_scores):,} items")
print(f"Confidence threshold C = {C:.1f} ratings")
print(f"Overall mean m = {m:.2f}")

# Show the top 10 most popular albums/artists/genres
top_items = item_stats.sort_values('bayesian_avg', ascending=False).head(10)
print("\nTop 10 globally popular items:")
print(top_items[['ItemID','global_mean','rating_count','bayesian_avg']].to_string(index=False))

Global popularity scores computed for 295,799 items
Confidence threshold C = 10.0 ratings
Overall mean m = 49.77

Top 10 globally popular items:
ItemID  global_mean  rating_count  bayesian_avg
251888        88.87           257         87.41
268359        89.30           115         86.14
169258        89.79            96         86.02
143794        90.01            76         85.33
264708        85.59           892         85.19
286359        84.74          2145         84.58
 39751        87.66           111         84.53
269021        89.55            66         84.31
258374        83.94           856         83.55
284170        84.65           226         83.17


In [12]:
# We can now use this global popularity scores to update our feature lookup

def get_score(item_id, user_ratings, global_item_scores, overall_mean):
    """
    3-level lookup for a single item:
      1. User's own rating           most personal signal
      2. Global Bayesian average     community popularity signal
      3. Overall dataset mean        last resort cold-start
    """
    if item_id is None:
        return overall_mean

    # User has rated this item directly
    if item_id in user_ratings:
        return user_ratings[item_id]

    # Global popularity (Bayesian average across all users)
    if item_id in global_item_scores:
        return global_item_scores[item_id]

    # item never rated by anyone
    return overall_mean

## b) Strategy Design ("Dig Deeper" Search Logic)

Even though we have a global fallback, there exists an issue:
If:
User has rated track 1 as 90 and track 2 as 85
both tracks belong to album 3
but the user has never reated album 3

Our code will fall back to the global average

So instead:
If a user loved 2 songs from an album, they will probably love the album. 
Their ratings for a track are insight towards their album preference

When we aggregate the scores for the tracks in an album to rank the album, we will use the mean. If we used the max, we would be evaluating "Whats the best song I have listened to from this album", which is an overestimation, and an underestimation when using the min. However, when using the mean, we are asking "What's my general feeling about this album's tracks", which is a fair estimate.

#### Build Album to Tracks Lookup 

In [13]:
# album_to_tracks: album_id (str) -> [track_id (str), ...]
album_to_tracks = defaultdict(list)
for _, row in tqdm(track_df.iterrows(), total=len(track_df), desc="Building album→tracks"):
    if pd.notna(row['AlbumID']):
        album_id  = str(int(row['AlbumID']))
        track_id  = str(int(row['TrackID']))
        album_to_tracks[album_id].append(track_id)
        
total_albums_with_tracks = len(album_to_tracks)
avg_tracks_per_album = np.mean([len(v) for v in album_to_tracks.values()])

print(f"Albums with at least one track: {total_albums_with_tracks:,}")
print(f"Average tracks per album      : {avg_tracks_per_album:.2f}")
print(f"Max tracks in one album       : {max(len(v) for v in album_to_tracks.values()):,}")

Building album→tracks:   0%|          | 0/224041 [00:00<?, ?it/s]

Albums with at least one track: 31,141
Average tracks per album      : 6.60
Max tracks in one album       : 142


#### Hierarchical Album Score

In [14]:
def get_album_score(album_id, user_ratings, album_to_tracks, global_item_scores, overall_mean, tracker):
    """
    3 levels of hierarchical lookup for album score:

    Primary     User's direct album rating
    Dig Deeper  Mean of user's ratings for tracks in the same album
    Global      Global Bayesian popularity of the album

    'tracker' is a dict that coun ts how often each tier is called.
    """
    if album_id is None:
        tracker['no_album_id'] += 1
        return overall_mean

    # Direct album rating 
    if album_id in user_ratings:
        tracker['tier1_direct'] += 1
        return user_ratings[album_id]

    # Dig Deeper — sibling track ratings
    sibling_tracks  = album_to_tracks.get(album_id, [])
    sibling_ratings = [user_ratings[t] for t in sibling_tracks if t in user_ratings]

    if sibling_ratings:
        tracker['tier2_dig_deeper'] += 1
        return np.mean(sibling_ratings)   # mean of known sibling tracks

    # Global popularity fallback
    tracker['tier3_global'] += 1
    return global_item_scores.get(album_id, overall_mean)

In [15]:
# Updated feature build with new fallbacks

def build_feature_vector_v2(user_id, track_id, track_to_hierarchy, user_to_ratings, album_to_tracks, global_item_scores, overall_mean, tracker):
    """
    Builds a 7-feature vector for a (user, track) pair.
    Album score now uses 3 hierarchical lookup.
    Artist and genre scores use 2-tier (direct and global).
    """
    uid = str(int(user_id))
    tid = str(int(track_id))

    hierarchy    = track_to_hierarchy.get(tid, {})
    user_ratings = user_to_ratings.get(uid, {})

    album_id  = hierarchy.get('album')
    artist_id = hierarchy.get('artist')
    genre_ids = hierarchy.get('genres', [])

    # Album score (3 levels: direct -> dig deeper -> global)
    album_score = get_album_score(album_id, user_ratings, album_to_tracks, global_item_scores, overall_mean, tracker)

    # Artist score (2 levels: direct -> global) 
    artist_score = get_score(artist_id, user_ratings, global_item_scores, overall_mean)

    # Genre scores (2-tier each: direct -> global) 
    genre_scores = [get_score(g, user_ratings, global_item_scores, overall_mean) for g in genre_ids]

    # Genre statistics
    if genre_scores:
        genre_count = len(genre_scores)
        genre_max   = float(np.max(genre_scores))
        genre_min   = float(np.min(genre_scores))
        genre_mean  = float(np.mean(genre_scores))
        genre_var   = float(np.var(genre_scores))
    else:
        genre_count = 0
        genre_max   = overall_mean
        genre_min   = overall_mean
        genre_mean  = overall_mean
        genre_var   = 0.0

    return {
        'album_score' : album_score,
        'artist_score': artist_score,
        'genre_count' : genre_count,
        'genre_max'   : genre_max,
        'genre_min'   : genre_min,
        'genre_mean'  : genre_mean,
        'genre_var'   : genre_var,
    }

In [16]:
# Build Feature Matrix and Track Fallback Usage

tracker = {
    'tier1_direct'   : 0,   # User rated the album directly
    'tier2_dig_deeper': 0,  # Inferred from sibling tracks
    'tier3_global'   : 0,   # Global popularity fallback
    'no_album_id'    : 0,   # Track has no album at all
}

records_v2 = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Building feature matrix v2"):
    fv = build_feature_vector_v2(
        user_id          = row['UserID'],
        track_id         = row['TrackID'],
        track_to_hierarchy = track_to_hierarchy,
        user_to_ratings  = user_to_ratings,
        album_to_tracks  = album_to_tracks,
        global_item_scores = global_item_scores,
        overall_mean     = overall_mean,
        tracker          = tracker,
    )
    fv['UserID']  = row['UserID']
    fv['TrackID'] = row['TrackID']
    records_v2.append(fv)

features_v2_df = pd.DataFrame(records_v2, columns=[
    'UserID', 'TrackID',
    'album_score', 'artist_score',
    'genre_count', 'genre_max', 'genre_min', 'genre_mean', 'genre_var'
])

print(f"Feature matrix shape: {features_v2_df.shape}")

Building feature matrix v2:   0%|          | 0/120000 [00:00<?, ?it/s]

Feature matrix shape: (120000, 9)


In [17]:
# Analyze how often each tier was called

total = sum(tracker.values())

print("=" * 50)
print("  ALBUM SCORE BREAKDOWN")
print("=" * 50)
for tier, count in tracker.items():
    pct = count / total * 100
    print(f"  {tier:<22}: {count:>7,}  ({pct:5.1f}%)")
print(f"  {'TOTAL':<22}: {total:>7,}")

  ALBUM SCORE BREAKDOWN
  tier1_direct          :  34,264  ( 28.6%)
  tier2_dig_deeper      :   5,678  (  4.7%)
  tier3_global          :  71,486  ( 59.6%)
  no_album_id           :   8,572  (  7.1%)
  TOTAL                 : 120,000


In [18]:
# Generate submissions using the improved feature matrix

features_v2_df['score_s1'] = features_v2_df.apply(score_strategy_1, axis=1)
features_v2_df['score_s2'] = features_v2_df.apply(score_strategy_2, axis=1)

submission_v2_s1 = generate_predictions(features_v2_df, 'score_s1')
submission_v2_s2 = generate_predictions(features_v2_df, 'score_s2')

submission_v2_s1.to_csv('submission_v2_strategy1.csv', index=False)
submission_v2_s2.to_csv('submission_v2_strategy2.csv', index=False)

print("Saved: submission_v2_strategy1.csv")
print("Saved: submission_v2_strategy2.csv")

Saved: submission_v2_strategy1.csv
Saved: submission_v2_strategy2.csv


Scores:
submission_v2_strategy1: 0.774
submission_v2_strategy2: 0.708

# Further Improvements

## Strategy 3: Adaptive Weight Normalization

Instead of filling missing album/artist/genre with fallback values, this approach focuses on including levels where the
user has included a rating, then normalizes the weights to sum to 1.0

Example:
If a user rates an album (90) but not the artist or genre:
- Current Strategy: 0.40*90 + 0.3*50 +0.3*50 = 66
- New Strategy    : (0.5*90)/0.5 = 90 (we give prefernce to the values we do have)

Cold-start (no direct signals at any level) falls back to overall_mean (50).
Weights: album=0.5, artist=0.3, genre=0.2  (album is the most specific signal)

In [19]:
def score_strategy_1_fixed(row, user_to_ratings, track_to_hierarchy, global_item_scores, overall_mean):
    """
    Same as Strategy 1 but: only fall back to Bayesian global if the user
    has NO direct ratings at any level. Otherwise use adaptive normalization.
    """
    uid = str(int(row['UserID']))
    tid = str(int(row['TrackID']))
    hierarchy    = track_to_hierarchy.get(tid, {})
    user_ratings = user_to_ratings.get(uid, {})

    album_id  = hierarchy.get('album')
    artist_id = hierarchy.get('artist')
    genre_ids = hierarchy.get('genres', [])

    score        = 0.0
    total_weight = 0.0

    # Primary: direct album rating
    # Secondary: infer album prefernce from sibling track ratings
    if album_id and album_id in user_ratings:
        score        += 0.40 * user_ratings[album_id]
        total_weight += 0.40
    elif album_id:
        sibling_tracks  = album_to_tracks.get(album_id, [])
        sibling_ratings = [user_ratings[t] for t in sibling_tracks if t in user_ratings]
        if sibling_ratings:
            score        += weights['album'] * np.mean(sibling_ratings)
            total_weight += weights['album']

    if artist_id and artist_id in user_ratings:
        score        += 0.30 * user_ratings[artist_id]
        total_weight += 0.30

    rated_genres = [user_ratings[g] for g in genre_ids if g in user_ratings]
    if rated_genres:
        score        += 0.30 * np.mean(rated_genres)
        total_weight += 0.30

    if total_weight > 0:
        return score / total_weight   # direct signal found —> normalize

    # Complete cold-start: use Bayesian global popularity as tiebreaker
    signals = []
    if album_id:
        signals.append(global_item_scores.get(album_id, overall_mean))
    if artist_id:
        signals.append(global_item_scores.get(artist_id, overall_mean))
    for g in genre_ids:
        signals.append(global_item_scores.get(g, overall_mean))
    return np.mean(signals) if signals else overall_mean

In [20]:
def score_strategy_3(user_id, track_id, track_to_hierarchy, user_to_ratings, overall_mean, 
weights={'album': 0.5, 'artist': 0.3, 'genre': 0.2}):

    uid = str(int(user_id))
    tid = str(int(track_id))

    hierarchy    = track_to_hierarchy.get(tid, {})
    user_ratings = user_to_ratings.get(uid, {})

    album_id  = hierarchy.get('album')
    artist_id = hierarchy.get('artist')
    genre_ids = hierarchy.get('genres', [])

    score        = 0.0
    total_weight = 0.0

    # Only include levels the user has DIRECTLY rated
    if album_id and album_id in user_ratings:
        score        += weights['album'] * user_ratings[album_id]
        total_weight += weights['album']

    if artist_id and artist_id in user_ratings:
        score        += weights['artist'] * user_ratings[artist_id]
        total_weight += weights['artist']

    rated_genres = [user_ratings[g] for g in genre_ids if g in user_ratings]
    if rated_genres:
        score        += weights['genre'] * np.mean(rated_genres)
        total_weight += weights['genre']

    if total_weight > 0:
        return score / total_weight
    # Complete cold-start, use Bayesian popularity as tiebreaker only
    signals = []
    if album_id:
        signals.append(global_item_scores.get(album_id, overall_mean))
    if artist_id:
        signals.append(global_item_scores.get(artist_id, overall_mean))
    for g in genre_ids:
        signals.append(global_item_scores.get(g, overall_mean))
    return np.mean(signals) if signals else overall_mean


predictions_s3 = []

for user_id, group in tqdm(test_df.groupby('UserID'), total=test_df['UserID'].nunique(), desc="Strategy 3"):
    uid    = str(int(user_id))
    tracks = group['TrackID'].tolist()

    scored = [(tid, score_strategy_3(user_id, tid, track_to_hierarchy, user_to_ratings, overall_mean))
              for tid in tracks]
    scored.sort(key=lambda x: x[1], reverse=True)

    for i, (tid, _) in enumerate(scored):
        predictions_s3.append({
            'TrackID'  : f"{user_id}_{tid}",
            'Predictor': 1 if i < 3 else 0
        })

submission_s3 = pd.DataFrame(predictions_s3)
submission_s3.to_csv('submission_v3_strategy3.csv', index=False)

submission_s3['uid'] = submission_s3['TrackID'].str.split('_').str[0]
errors = (submission_s3.groupby('uid')['Predictor'].sum() != 3).sum()
print(f"Validation errors: {errors}")
print("Saved: submission_v3_strategy3.csv")

Strategy 3:   0%|          | 0/20000 [00:00<?, ?it/s]

Validation errors: 0
Saved: submission_v3_strategy3.csv


Score: 0.792

## Strategy 5: S3 + Genre Max-Blend + Track-Level Cold-Start

### Two targeted improvements to S3's weak points

**1. Genre signal: max-blend instead of plain mean**
S3 uses `mean(rated_genres)`. If a user rated Genre X = 90 and Genre Y = 20,
the mean is 55 — "meh". But the track *has* a genre the user loves. The
max-blend `0.6 * max + 0.4 * mean` = 76 correctly rewards the "gateway genre."
When all genre scores are similar, max ≈ mean so the blend behaves like S3.

**2. Track-level cold-start**
When a user has NO direct hierarchy signals (pure cold-start), S3 falls back
to a flat mean of Bayesian global scores. But `global_item_scores` already
contains track-level Bayesian scores (train_data uses the flat ItemID namespace,
so many TrackIDs appear as rated ItemIDs). The track's own global score is the
most specific popularity signal available — more discriminative than album or
genre-level scores for ordering tracks we know nothing personal about.
Cold-start ranking: track global (0.50) → album (0.25) → artist (0.15) → genres (0.10)
When track has no global score, falls back to: album (0.50) → artist (0.30) → genres (0.20)

In [31]:
def score_strategy_5(user_id, track_id):
    """
    S3 core (direct-only adaptive normalization) with two improvements:

    1. Genre signal = 0.6 * max(rated) + 0.4 * mean(rated)
       Rewards a strongly-loved gateway genre instead of averaging it down.
       When all genre scores are similar, behaves identically to S3.

    2. Cold-start uses track's own global Bayesian score at highest weight (0.50).
       The track may appear as a rated ItemID in train_data even if this user
       never rated it — that community signal is more specific than album/genre.
    """
    uid = str(int(user_id))
    tid = str(int(track_id))

    hierarchy    = track_to_hierarchy.get(tid, {})
    user_ratings = user_to_ratings.get(uid, {})

    album_id  = hierarchy.get('album')
    artist_id = hierarchy.get('artist')
    genre_ids = hierarchy.get('genres', [])

    score        = 0.0
    total_weight = 0.0

    # --- Album: direct rating only -------------------------------------------
    if album_id and album_id in user_ratings:
        score        += 0.50 * user_ratings[album_id]
        total_weight += 0.50

    # --- Artist: direct rating only ------------------------------------------
    if artist_id and artist_id in user_ratings:
        score        += 0.30 * user_ratings[artist_id]
        total_weight += 0.30

    # --- Genre: direct ratings only, max-blend signal ------------------------
    # 0.6 * max captures the "gateway genre" (one loved genre = likely liked)
    # 0.4 * mean prevents a single outlier from dominating when multiple genres rated
    rated_genre_scores = [user_ratings[g] for g in genre_ids if g in user_ratings]
    if rated_genre_scores:
        genre_signal = 0.6 * max(rated_genre_scores) + 0.4 * np.mean(rated_genre_scores)
        score        += 0.20 * genre_signal
        total_weight += 0.20

    # --- Normalize over weights that actually fired --------------------------
    if total_weight > 0:
        return score / total_weight

    # --- Cold-start: specificity-weighted Bayesian with track-level signal ---
    # Track's own global score is the most discriminative for pure cold-start users.
    # Weights normalise automatically via cs_weight division.
    track_global = global_item_scores.get(tid)

    cs_score  = 0.0
    cs_weight = 0.0

    if track_global is not None:
        cs_score  += 0.50 * track_global
        cs_weight += 0.50

    if album_id:
        cs_score  += 0.25 * global_item_scores.get(album_id, overall_mean)
        cs_weight += 0.25

    if artist_id:
        cs_score  += 0.15 * global_item_scores.get(artist_id, overall_mean)
        cs_weight += 0.15

    if genre_ids:
        # Spread genre weight evenly so genre count doesn't skew the result
        genre_w = 0.10 / len(genre_ids)
        for g in genre_ids:
            cs_score  += genre_w * global_item_scores.get(g, overall_mean)
            cs_weight += genre_w

    return cs_score / cs_weight if cs_weight > 0 else overall_mean

In [32]:
# Strategy 5 — run on full test set and generate submission

predictions_s5 = []

for user_id, group in tqdm(
    test_df.groupby('UserID'),
    total=test_df['UserID'].nunique(),
    desc="Strategy 5"
):
    tracks = group['TrackID'].tolist()

    scored = [
        (tid, score_strategy_5(user_id, tid))
        for tid in tracks
    ]
    scored.sort(key=lambda x: x[1], reverse=True)

    for i, (tid, _) in enumerate(scored):
        predictions_s5.append({
            'TrackID'  : f"{user_id}_{tid}",
            'Predictor': 1 if i < 3 else 0
        })

submission_s5 = pd.DataFrame(predictions_s5)
submission_s5.to_csv('submission_v5_strategy5.csv', index=False)

# Validation
submission_s5['uid'] = submission_s5['TrackID'].str.split('_').str[0]
errors = (submission_s5.groupby('uid')['Predictor'].sum() != 3).sum()
print(f"Validation errors : {errors}")
print(f"Total predictions : {len(submission_s5):,}")
print("Saved: submission_v5_strategy5.csv")

# Diagnostic: how often does the track-level global score exist for cold-start users?
cold_start_with_track = sum(
    1 for _, row in test_df.iterrows()
    if user_to_ratings.get(str(int(row['UserID'])), {}) == {}
    and global_item_scores.get(str(int(row['TrackID']))) is not None
)
print(f"\nCold-start rows with track-level Bayesian score: {cold_start_with_track:,}")

Strategy 5:   0%|          | 0/20000 [00:00<?, ?it/s]

Validation errors : 0
Total predictions : 120,000
Saved: submission_v5_strategy5.csv

Cold-start rows with track-level Bayesian score: 0


In [33]:
Score: 0.779